# Day 1 Runner — `llm-from-base-to-assistant`

**Setup + Foundations for Qwen3-1.7B-Base**, meant to run on a **rented GPU**
(Colab, RunPod, Lambda, Vast, etc.). No training happens today — you set up the
environment, confirm the base model runs on your GPU, explore the model's
internals, capture "before" evidence, and lock the held-out eval set.

**How to use:** run the cells top to bottom. Each section maps to one Day 1 script.
Read the short note above each cell — that's the concept it demonstrates.

> If you hit an out-of-memory error in the smoke test, switch the model to the
> `Qwen/Qwen3-0.6B-Base` dev mule in `configs/day1.yaml` (a cell below shows how)
> and re-run.

## 0. Check the GPU

Confirm a GPU is actually attached before doing anything else.

In [ ]:
!nvidia-smi

## 1. Get the project code

Two options — pick ONE.

**Option A — clone from GitHub** (if you've pushed the repo). Replace the URL.

**Option B — upload the repo zip** to the machine and unzip it (use this if you
haven't pushed to GitHub yet).

In [ ]:
# --- Option A: clone from GitHub (edit the URL) ---
# !git clone https://github.com/<your-username>/llm-from-base-to-assistant.git
# %cd llm-from-base-to-assistant

# --- Option B: unzip an uploaded archive ---
# On Colab: use the file browser to upload llm-from-base-to-assistant.zip first.
# !unzip -q llm-from-base-to-assistant.zip
# %cd llm-from-base-to-assistant

import os
print("current dir:", os.getcwd())
print("contents:", os.listdir())

## 2. Install the pinned dependencies

Uses the repo's `requirements.txt` so versions match the rest of the project.
On a fresh rented box this takes a few minutes.

> On Colab, `torch` is usually preinstalled with the right CUDA build — if the
> pinned torch version conflicts, comment out the torch line in
> `requirements.txt` and keep Colab's build.

In [ ]:
!pip install -q -r requirements.txt
print("\nInstall done.")

## 3. Environment sanity test (no model download)

Fast checks: the pinned stack imports, the config parses, the base model is not
accidentally the instruct model, and the repo skeleton + scripts exist.

In [ ]:
!python -m pytest tests/test_environment.py -q

## (optional) Switch to the 0.6B dev mule

Only run this cell if the smoke test below OOMs, or if you want to debug cheaply.
It rewrites the model id in `configs/day1.yaml`. Re-run later with the 1.7B model
for the real work.

In [ ]:
# import re, pathlib
# p = pathlib.Path("configs/day1.yaml"); s = p.read_text()
# s = s.replace('id: "Qwen/Qwen3-1.7B-Base"', 'id: "Qwen/Qwen3-0.6B-Base"')
# p.write_text(s); print("Switched config to Qwen3-0.6B-Base")

## 4. Smoke test — does the model run, and what does it cost?

Loads `Qwen3-1.7B-Base` (first download happens here — a few GB), generates a
short continuation, and prints **measured peak VRAM** and **tokens/sec**.
This is the "measure, don't assume" step. Note these two numbers.

In [ ]:
!python scripts/smoke_test.py --config configs/day1.yaml

## 5. Explore the tokenizer

See how text becomes tokens/IDs, what the special tokens (EOS/PAD) are, and how
token counts differ between tokenizers (why cost is measured in tokens).

Feel free to edit `SAMPLES` inside `scripts/explore_tokenizer.py` to use your own
LLM-domain text.

In [ ]:
!python scripts/explore_tokenizer.py --config configs/day1.yaml

## 6. Inspect the model architecture

Prints the module structure and counts parameters per component (embeddings vs
attention vs MLP vs LM head). Confirms the theory: the embedding table's share,
GQA (KV heads < query heads), and tied embeddings.

In [ ]:
!python scripts/inspect_model.py --config configs/day1.yaml

## 7. One forward pass — top-10 next-token probabilities

Turns "logits → softmax → distribution" into something you can see: for a prompt,
what does the model think comes next, and how confident is it?

In [ ]:
!python scripts/forward_pass_probs.py --config configs/day1.yaml --prompt "The capital of France is" 

## 8. Compare decoding settings (temperature 0 / 0.7 / 1.2)

Feel what the decoding dials do: greedy is deterministic and flat; higher
temperature is varied and riskier.

In [ ]:
!python scripts/generation_settings.py --config configs/day1.yaml --prompt "Explain what a tokenizer does." 

## 9. Capture "before" evidence + lock the eval set

**9a.** Show the base model *continuing text instead of answering* — your "before"
evidence for the Day 3 before/after. Saves to `docs/before_evidence.md`.

In [ ]:
!python scripts/base_vs_chat.py --config configs/day1.yaml --save docs/before_evidence.md

**9b.** Lock the held-out evaluation folder with its guard README — the one
hard rule: never train on this data.

In [ ]:
!python scripts/lock_eval_set.py

## 10. Record your decisions

Open `docs/decisions.md` and fill in the hardware table (model, GPU, the measured
peak VRAM and tokens/sec from step 4) and the LoRA-vs-full-FT choice. This cell
just prints the file so you can see what to complete.

In [ ]:
print(open("docs/decisions.md").read())

## ✅ Day 1 complete

You now have: a working pinned environment, the base model confirmed running on
your GPU (with measured VRAM/throughput), the model's internals explored,
"before" evidence saved, and the held-out eval set locked.

**Next — Day 2:** collect the LLM-domain corpus, clean/dedup/pack it, split it,
and run continued pretraining (CPT).

> If you cloned from GitHub and want to keep your `before_evidence.md` and filled
> `decisions.md`, commit and push them before the rented machine is torn down:
>
> ```
> !git add docs/before_evidence.md docs/decisions.md
> !git commit -m "Day 1: results + decisions"
> !git push
> ```